In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Trips

In [3]:
def total_trips(data1, data2, tag='PSRC Region'):
    Trip_1_total = get_total(data1['Trip']['trexpfac'])
    Trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])

    tpp  = pd.DataFrame(index = ['Trip'])
    tpp['DaysimOutputs'] = Trip_1_total
    tpp[f'{survey_year}Survey'] = Trip_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.0f}',
        f'{survey_year}Survey': '{:,.0f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.0f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [4]:
total_trips(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,"16,180,996","13,902,503","2,278,493",16.4%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_trips(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,"1,273,757","853,837","419,920",49.2%


## Trips per Person

In [6]:
def trip_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Trip_1_total = get_total(data1['Trip']['trexpfac'])
    Trip_2_total = get_total(data2['Trip_cloned']['trexpfac'])

    ##Trips per person
    tpp1 = Trip_1_total / Person_1_total
    tpp2 = Trip_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Trip'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [7]:
trip_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,3.8,3.7,0.1,2.2%


In [8]:
trip_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Trip,3.9,3.0,1.0,33.0%


## Trips per Person by Purpose

In [9]:
def trips_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trips per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Trip'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_1_total
    tpbp2 = data2['Trip_cloned'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Trips per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Trips per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'dpurp', 'Trip Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Trips per Person (DaysimOutputs)': '{:,.1f}',
        f'Trips per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Trip Purpose',
        y=['Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Trips per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [10]:
trips_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Purpose,,,,
Work,0.5,0.5,-0.0,-3.0%
School,0.1,0.1,0.0,3.7%
Escort,0.4,0.4,-0.0,-1.3%
Personal Business,0.2,0.2,-0.0,-14.9%
Shop,0.5,0.5,-0.0,-1.5%
Meal,0.2,0.2,0.0,3.4%
Social,0.5,0.5,0.1,9.6%


In [11]:
trips_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Purpose,,,,
Work,0.5,0.4,0.1,24.2%
School,0.2,0.1,0.0,16.9%
Escort,0.4,0.3,0.1,17.0%
Personal Business,0.2,0.1,0.0,22.4%
Shop,0.5,0.2,0.2,110.1%
Meal,0.3,0.3,0.0,10.1%
Social,0.6,0.5,0.2,37.1%


## Trips per Person by Mode

In [12]:
def trips_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trips per Person by Mode
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Trip'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / Person_1_total
    tpbp2 = data2['Trip_cloned'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Trips per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Trips per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'mode', 'Trip Mode')
    tpbp = tpbp.loc[trip_mode_cat.values()]
    # table
    display(tpbp.style.format({
        'Trips per Person (DaysimOutputs)': '{:,.1f}',
        f'Trips per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Trips per Person (DaysimOutputs) - Trips per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Trip Mode',
        y=['Trips per Person (DaysimOutputs)', f'Trips per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Trips per Person by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()    

In [13]:
trips_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Mode,,,,
Walk,0.6,0.4,0.1,36.7%
Bike,0.0,0.0,-0.0,-79.9%
SOV,1.8,1.7,0.1,7.8%
HOV2,0.9,0.8,0.1,13.5%
HOV3+,0.4,0.6,-0.2,-28.7%
Transit,0.1,0.1,-0.0,-40.8%
School Bus,0.0,0.1,-0.0,-30.2%


In [14]:
trips_per_ps_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Trip Mode,,,,
Walk,0.5,0.4,0.1,18.8%
Bike,0.0,0.0,-0.0,-68.2%
SOV,1.9,1.2,0.7,58.9%
HOV2,1.0,0.7,0.3,38.7%
HOV3+,0.4,0.4,0.0,9.1%
Transit,0.1,0.1,-0.0,-17.0%
School Bus,0.1,0.1,-0.0,-26.8%


## Trip Share by Purpose

In [15]:
def pc_trip_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Trips by Purpose
    trip_1_total = get_total(data_daysim['Trip']['trexpfac'])
    trip_2_total = get_total(data_survey['Trip_cloned']['trexpfac'])
    ptbp1 = 100 * data1['Trip'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / trip_1_total
    ptbp2 = 100 * data2['Trip_cloned'][['dpurp','trexpfac']].groupby('dpurp').sum()['trexpfac'] / trip_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Trips (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Trips ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'dpurp', 'Trips Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Trips (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Trips ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Trips Purpose',
        y=['Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Trips by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Trips', xaxis_title='Trips Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [16]:
pc_trip_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Purpose,,,,
Work,12.6%,13.2%,-0.7%,-5.1%
School,3.9%,3.9%,0.1%,1.5%
Escort,9.5%,9.9%,-0.3%,-3.5%
Personal Business,4.6%,5.5%,-0.9%,-16.8%
Shop,12.9%,13.4%,-0.5%,-3.7%
Meal,6.7%,6.6%,0.1%,1.1%
Social,14.4%,13.4%,1.0%,7.2%


In [17]:
pc_trip_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Purpose,,,,
Work,0.9%,0.8%,0.1%,19.7%
School,0.3%,0.3%,0.0%,12.7%
Escort,0.7%,0.6%,0.1%,12.8%
Personal Business,0.3%,0.3%,0.1%,17.9%
Shop,0.9%,0.5%,0.5%,102.5%
Meal,0.6%,0.6%,0.0%,6.1%
Social,1.2%,0.9%,0.3%,32.2%


## Trip Share by Mode

In [18]:
def pc_trip_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Trips by Mode
    trip_1_total = get_total(data_daysim['Trip']['trexpfac'])
    trip_2_total = get_total(data_survey['Trip_cloned']['trexpfac'])
    ptbp1 = 100 * data1['Trip'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / trip_1_total
    ptbp2 = 100 * data2['Trip_cloned'][['mode','trexpfac']].groupby('mode').sum()['trexpfac'] / trip_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Trips (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Trips ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'mode', 'Trips Mode')
    ptbp = ptbp.loc[trip_mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Trips (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Trips ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Trips Mode',
        y=['Percent of Trips (DaysimOutputs)', f'Percent of Trips ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Trips by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Trips', xaxis_title='Trips Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [19]:
pc_trip_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Mode,,,,
Walk,14.9%,11.2%,3.8%,33.7%
Bike,0.2%,1.2%,-1.0%,-80.3%
SOV,48.1%,45.6%,2.5%,5.5%
HOV2,23.2%,20.9%,2.3%,11.0%
HOV3+,10.7%,15.3%,-4.7%,-30.3%
Transit,1.7%,3.0%,-1.2%,-42.1%
School Bus,1.1%,1.6%,-0.5%,-31.7%


In [20]:
pc_trip_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Trips (DaysimOutputs),Percent of Trips (2023Survey),Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey)),% Difference (Percent of Trips (DaysimOutputs) - Percent of Trips (2023Survey))
Trips Mode,,,,
Walk,1.0%,0.9%,0.1%,14.5%
Bike,0.0%,0.1%,-0.0%,-69.4%
SOV,3.8%,2.5%,1.3%,53.1%
HOV2,1.9%,1.4%,0.5%,33.6%
HOV3+,0.8%,0.8%,0.0%,5.1%
Transit,0.2%,0.3%,-0.1%,-20.0%
School Bus,0.1%,0.1%,-0.0%,-29.5%


## Trip Distance by Purpose

In [21]:
def trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.1f}',
        f'Average Trip Length ({name2})': '{:,.1f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [22]:
trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Work,9.0,9.2,-0.2,-2.3%
School,4.7,3.9,0.8,20.4%
Escort,5.9,5.7,0.2,2.8%
Personal Business,5.2,5.6,-0.4,-6.4%
Shop,4.3,4.4,-0.1,-2.0%
Meal,3.8,3.3,0.5,13.5%
Social,5.2,6.7,-1.5,-22.1%


In [23]:
trips_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Work,7.6,6.7,0.9,13.2%
School,3.8,4.2,-0.4,-9.2%
Escort,4.7,7.5,-2.7,-36.3%
Personal Business,5.0,4.2,0.8,18.4%
Shop,3.0,4.2,-1.2,-27.6%
Meal,3.1,2.9,0.2,7.5%
Social,4.3,6.4,-2.0,-31.9%


## Trip Distance by Mode

In [24]:
def trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by mode
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Mode')  
    atl = atl.loc[trip_mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.1f}',
        f'Average Trip Length ({name2})': '{:,.1f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Mode',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [25]:
trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Mode,,,,
Walk,0.9,0.9,-0.0,-0.1%
Bike,4.6,2.0,2.6,129.0%
SOV,7.0,6.9,0.1,1.5%
HOV2,5.9,6.2,-0.3,-5.6%
HOV3+,6.8,7.3,-0.6,-8.0%
Transit,9.7,8.9,0.8,8.8%
School Bus,3.9,3.6,0.2,6.3%


In [26]:
trips_distance_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Mode,,,,
Walk,1.4,0.7,0.7,101.5%
Bike,6.7,1.5,5.3,356.2%
SOV,5.4,6.2,-0.8,-13.1%
HOV2,4.5,8.5,-4.0,-47.6%
HOV3+,4.5,5.4,-0.9,-16.6%
Transit,5.5,10.2,-4.8,-46.5%
School Bus,2.8,3.1,-0.3,-11.2%


## Trip Travel Time by Purpose

In [27]:
def trips_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by trip purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travtime', 'trexpfac', 'dpurp']], 'travtime', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travtime', 'trexpfac', 'dpurp']], 'travtime', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Travel Time (' + name1 + ')', 'Average Trip Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Trip Travel Time ({name1})': '{:,.1f}',
        f'Average Trip Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Travel Time ({name1})', f'Average Trip Travel Time ({name2})'],
        barmode='group',
        title=f'Average Trip Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Travel Time', xaxis_title='Trip Purpose', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [28]:
trips_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Purpose,,,,
Work,24.5,21.6,2.8,13.1%
School,18.6,12.4,6.2,49.7%
Escort,19.2,14.7,4.5,30.8%
Personal Business,18.6,21.6,-3.0,-14.1%
Shop,16.9,12.7,4.2,33.6%
Meal,15.2,12.4,2.8,22.3%
Social,18.8,17.2,1.5,8.8%


In [29]:
trips_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Purpose,,,,
Work,18.1,17.8,0.3,1.6%
School,14.7,11.8,2.9,24.3%
Escort,13.1,17.2,-4.2,-24.1%
Personal Business,14.3,13.2,1.2,8.9%
Shop,11.2,12.0,-0.9,-7.2%
Meal,11.1,9.3,1.8,18.9%
Social,14.1,19.2,-5.2,-26.8%


## Trip Travel Time by Mode

In [30]:
def trips_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by trip mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by mode
    triptotal1 = weighted_average(trip_ok_1[['travtime', 'trexpfac', 'mode']], 'travtime', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travtime', 'trexpfac', 'mode']], 'travtime', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Travel Time (' + name1 + ')', 'Average Trip Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[trip_mode_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Trip Travel Time ({name1})': '{:,.1f}',
        f'Average Trip Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Travel Time ({name1}) - Average Trip Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Mode',
        y=[f'Average Trip Travel Time ({name1})', f'Average Trip Travel Time ({name2})'],
        barmode='group',
        title=f'Average Trip Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Travel Time', xaxis_title='Trip Mode', xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    fig.show()

In [31]:
trips_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Mode,,,,
Walk,18.1,18.2,-0.1,-0.7%
Bike,30.7,12.1,18.6,153.9%
SOV,20.3,16.1,4.1,25.6%
HOV2,18.0,14.5,3.5,24.4%
HOV3+,19.8,16.3,3.5,21.8%
Transit,29.4,56.0,-26.6,-47.6%
School Bus,12.8,10.1,2.6,25.8%


In [32]:
trips_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Trip Travel Time (DaysimOutputs),Average Trip Travel Time (2023Survey),Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey)),% Difference (Average Trip Travel Time (DaysimOutputs) - Average Trip Travel Time (2023Survey))
Trip Mode,,,,
Walk,28.8,14.4,14.4,100.0%
Bike,44.9,8.9,36.0,405.7%
SOV,11.8,15.1,-3.3,-21.7%
HOV2,9.7,17.6,-8.0,-45.1%
HOV3+,9.5,13.4,-3.9,-28.8%
Transit,25.4,27.8,-2.4,-8.6%
School Bus,6.3,8.5,-2.2,-25.5%


## Trips by District

In [33]:
##Trips per Person by Purpose and Person Type/Number of Stops
data1=data_daysim
data2=data_survey
name1 = 'DaysimOutputs'
name2 = f'{survey_year}Survey' 
# calculate the percentage of each number of stops by purpose
data1['Household'] = data1['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data1['Trip'] = data1['Trip'].merge(data1['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
data2['Household'] = data2['Household'].merge(taz_subarea[['TAZ', 'DistrictFlowName']], left_on='hhtaz', right_on='TAZ', how='left')
data2['Trip_cloned'] = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'DistrictFlowName', 'hhincome', 'hhvehs']], on='hhno', how='left')
todist1 = data1['Trip'].groupby(by='DistrictFlowName')['trexpfac'].sum()
todist2 = data2['Trip_cloned'].groupby(by='DistrictFlowName')['trexpfac'].sum()
df_compare = pd.concat([todist1, todist2], axis=1)
df_compare.columns = [name1, name2]
df_compare['Difference'] = df_compare[name1] - df_compare[name2]
df_compare['% Difference'] = (df_compare['Difference'] / df_compare[name2]) * 100
display(df_compare.loc[district_flow_name.values()].style.format({
    name1: '{:,.0f}',
    name2: '{:,.0f}',
    'Difference': '{:,.0f}',
    '% Difference': '{:,.1f}%'
}))

,DaysimOutputs,2023Survey,Difference,% Difference
DistrictFlowName,,,,
Bellevue (excluding downtown),"539,077","426,535","112,542",26.4%
Bellevue Downtown,"72,397","19,581","52,816",269.7%
Kirkland,"371,216","156,492","214,724",137.2%
Redmond,"291,067","251,229","39,838",15.9%
Seattle (excluding Seattle downtown),"2,826,156","2,052,351","773,805",37.7%
Seattle downtown,"397,716","221,974","175,742",79.2%
Rest,"11,683,367","8,337,507","3,345,860",40.1%


In [34]:
def trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Trip Purpose',
                        gp1_list=[], gp2_list=[]):
    trip_by_district_purpose1 = data1['Trip'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    trip_by_district_purpose2 = data2['Trip_cloned'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    trip_by_district_purpose1 = trip_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose2 = trip_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose1.columns.name = gp2_label
    trip_by_district_purpose2.columns.name = gp2_label
    trip_by_district_purpose1.index.name = gp1_label
    trip_by_district_purpose2.index.name = gp1_label
    display(trip_by_district_purpose1.style.format('{:,.0f}').set_caption("DaysimOutputs"))
    display(trip_by_district_purpose2.style.format('{:,.0f}').set_caption(f"{survey_year}Survey"))
    percent_diff = (trip_by_district_purpose1 - trip_by_district_purpose2) / trip_by_district_purpose2 * 100
    display(percent_diff.style.format('{:,.1f}%').set_caption("Percentage Difference (DaysimOutputs - Survey)"))

In [35]:
def trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='DistrictFlowName', gp2='pdpurp', 
                        gp1_label='DistrictFlowName', gp2_label='Trip Purpose',
                        gp1_list=[], gp2_list=[]):
    trip_by_district_purpose1 = data1['Trip'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    trip_by_district_purpose2 = data2['Trip_cloned'].groupby([gp1, gp2], observed=False)['trexpfac'].sum().unstack(gp2)
    # Reorder index and columns as requested
    trip_by_district_purpose1 = trip_by_district_purpose1.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose2 = trip_by_district_purpose2.loc[list(gp1_list), list(gp2_list)]
    trip_by_district_purpose1.columns.name = gp2_label
    trip_by_district_purpose2.columns.name = gp2_label
    trip_by_district_purpose1.index.name = gp1_label
    trip_by_district_purpose2.index.name = gp1_label
    trip_by_district_purpose1 = trip_by_district_purpose1.div(trip_by_district_purpose1.sum(axis=0), axis=1) * 100
    trip_by_district_purpose2 = trip_by_district_purpose2.div(trip_by_district_purpose2.sum(axis=0), axis=1) * 100
    display(trip_by_district_purpose1.style.format('{:,.1f}%').set_caption("DaysimOutputs"))
    display(trip_by_district_purpose2.style.format('{:,.1f}%').set_caption(f"{survey_year}Survey"))

## Trips by District by Purpose

In [36]:
trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='dpurp', gp2='DistrictFlowName', 
                        gp1_label='Trip Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,"58,597","7,879","45,210","35,558","419,390","72,964","1,393,661"
School,"27,089","1,975","15,029","11,802","109,725","6,291","461,701"
Escort,"51,501","4,260","34,100","26,929","234,879","19,063","1,173,120"
Personal Business,"22,341","3,809","16,123","12,335","122,734","18,247","544,484"
Shop,"61,616","9,508","43,003","33,237","314,294","43,946","1,584,314"
Meal,"42,117","6,767","29,100","22,959","221,839","36,190","721,653"
Social,"84,113","11,658","57,606","45,142","414,055","57,339","1,661,237"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,"48,404","2,105","22,413","32,729","284,045","24,064","1,071,449"
School,"17,564",485,"6,663","17,917","87,425","1,864","373,969"
Escort,"52,885",309,"16,923","18,872","214,830","17,988","840,813"
Personal Business,"23,743","1,357","6,131","8,556","86,163","9,377","503,748"
Shop,"29,946","2,997","20,814","8,757","226,225","32,539","1,231,062"
Meal,"54,027","2,906","12,741","12,070","128,974","23,421","479,709"
Social,"61,144","2,459","13,513","51,942","286,611","46,249","1,020,379"


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,21.1%,274.4%,101.7%,8.6%,47.6%,203.2%,30.1%
School,54.2%,307.4%,125.6%,-34.1%,25.5%,237.5%,23.5%
Escort,-2.6%,"1,276.8%",101.5%,42.7%,9.3%,6.0%,39.5%
Personal Business,-5.9%,180.8%,163.0%,44.2%,42.4%,94.6%,8.1%
Shop,105.8%,217.2%,106.6%,279.6%,38.9%,35.1%,28.7%
Meal,-22.0%,132.9%,128.4%,90.2%,72.0%,54.5%,50.4%
Social,37.6%,374.1%,326.3%,-13.1%,44.5%,24.0%,62.8%


In [37]:
trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                        gp1='dpurp', gp2='DistrictFlowName', 
                        gp1_label='Trip Purpose', gp2_label='DistrictFlowName',
                        gp1_list=pdpurp_cat.values(), gp2_list=district_flow_name.values())

DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,16.9%,17.2%,18.8%,18.9%,22.8%,28.7%,18.5%
School,7.8%,4.3%,6.3%,6.3%,6.0%,2.5%,6.1%
Escort,14.8%,9.3%,14.2%,14.3%,12.8%,7.5%,15.6%
Personal Business,6.4%,8.3%,6.7%,6.6%,6.7%,7.2%,7.2%
Shop,17.7%,20.7%,17.9%,17.7%,17.1%,17.3%,21.0%
Meal,12.1%,14.8%,12.1%,12.2%,12.1%,14.2%,9.6%
Social,24.2%,25.4%,24.0%,24.0%,22.5%,22.6%,22.0%


DistrictFlowName,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest
Trip Purpose,,,,,,,
Work,16.8%,16.7%,22.6%,21.7%,21.6%,15.5%,19.4%
School,6.1%,3.8%,6.7%,11.9%,6.7%,1.2%,6.8%
Escort,18.4%,2.5%,17.1%,12.5%,16.3%,11.6%,15.2%
Personal Business,8.3%,10.8%,6.2%,5.7%,6.6%,6.0%,9.1%
Shop,10.4%,23.8%,21.0%,5.8%,17.2%,20.9%,22.3%
Meal,18.8%,23.0%,12.8%,8.0%,9.8%,15.1%,8.7%
Social,21.3%,19.5%,13.6%,34.4%,21.8%,29.7%,18.5%


## Trips by District by Mode

In [38]:
trips_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='mode', 
                    gp1_label='DistrictFlowName', gp2_label='Trip Mode',
                    gp1_list=district_flow_name.values(), gp2_list=trip_mode_cat.values())

Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),"57,033","1,032","256,461","138,414","60,338","17,417","8,382"
Bellevue Downtown,"32,155",550,"21,899","10,727","3,021","3,612",433
Kirkland,"38,595",652,"188,043","93,085","37,080","9,732","4,029"
Redmond,"34,119",721,"144,753","70,617","29,488","8,180","3,189"
Seattle (excluding Seattle downtown),"700,433","9,114","1,261,569","515,383","212,522","102,672","24,463"
Seattle downtown,"251,016","2,035","88,825","29,417","8,841","17,175",407
Rest,"1,305,321","24,598","5,822,816","2,899,225","1,380,745","118,233","132,429"


Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),"76,702","4,485","147,254","128,070","54,563","7,128","2,265"
Bellevue Downtown,"4,578",442,"7,211","5,947",523,694,nan
Kirkland,"23,580",nan,"83,706","21,964","17,462","5,945","1,992"
Redmond,"16,671","3,366","104,779","45,137","33,636","28,039","15,273"
Seattle (excluding Seattle downtown),"393,006","76,872","836,632","433,242","191,892","96,273","13,117"
Seattle downtown,"89,162","7,743","49,583","45,357","9,335","14,781",55
Rest,"556,911","38,913","3,946,708","1,796,021","1,514,071","206,229","154,794"


Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),-25.6%,-77.0%,74.2%,8.1%,10.6%,144.4%,270.1%
Bellevue Downtown,602.4%,24.5%,203.7%,80.4%,478.0%,420.1%,nan%
Kirkland,63.7%,nan%,124.6%,323.8%,112.3%,63.7%,102.3%
Redmond,104.7%,-78.6%,38.2%,56.5%,-12.3%,-70.8%,-79.1%
Seattle (excluding Seattle downtown),78.2%,-88.1%,50.8%,19.0%,10.8%,6.6%,86.5%
Seattle downtown,181.5%,-73.7%,79.1%,-35.1%,-5.3%,16.2%,640.3%
Rest,134.4%,-36.8%,47.5%,61.4%,-8.8%,-42.7%,-14.4%


In [39]:
trip_share_by_gp1_by_gp2(data1=data_daysim, data2=data_survey, 
                    gp1='DistrictFlowName', gp2='mode', 
                    gp1_label='DistrictFlowName', gp2_label='Trip Mode',
                    gp1_list=district_flow_name.values(), gp2_list=trip_mode_cat.values())

Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),2.4%,2.7%,3.3%,3.7%,3.5%,6.3%,4.8%
Bellevue Downtown,1.3%,1.4%,0.3%,0.3%,0.2%,1.3%,0.2%
Kirkland,1.6%,1.7%,2.4%,2.5%,2.1%,3.5%,2.3%
Redmond,1.4%,1.9%,1.9%,1.9%,1.7%,3.0%,1.8%
Seattle (excluding Seattle downtown),29.0%,23.5%,16.2%,13.7%,12.3%,37.1%,14.1%
Seattle downtown,10.4%,5.3%,1.1%,0.8%,0.5%,6.2%,0.2%
Rest,54.0%,63.6%,74.8%,77.2%,79.7%,42.7%,76.4%


Trip Mode,Walk,Bike,SOV,HOV2,HOV3+,Transit,School Bus
DistrictFlowName,,,,,,,
Bellevue (excluding downtown),6.6%,3.4%,2.8%,5.2%,3.0%,2.0%,1.2%
Bellevue Downtown,0.4%,0.3%,0.1%,0.2%,0.0%,0.2%,nan%
Kirkland,2.0%,nan%,1.6%,0.9%,1.0%,1.7%,1.1%
Redmond,1.4%,2.6%,2.0%,1.8%,1.8%,7.8%,8.1%
Seattle (excluding Seattle downtown),33.9%,58.3%,16.2%,17.5%,10.5%,26.8%,7.0%
Seattle downtown,7.7%,5.9%,1.0%,1.8%,0.5%,4.1%,0.0%
Rest,48.0%,29.5%,76.3%,72.5%,83.1%,57.4%,82.6%
